# 04 — Prototype: location → 12-month LULC change

**Input:** latitude, longitude, start month (+ year). **Output:** 12 monthly LULC
maps, per-class area %, month-to-month + total change (percentage points), a class
**transition matrix** (A→B), a spatial **change map**, a **confidence map**, and an
independent **Google Dynamic World** sanity check.

**Pipeline:** pull 12 monthly Sentinel-1 composites (same preprocessing as training,
bundled into one Earth Engine request) → U-Net per month → temporal LSTM → analysis.
Full code in `prototype.py` (CLI) and `app.py` (interactive Gradio UI).

In [ ]:
# How % and change are computed
import numpy as np
def class_pct(map_hw, n=7):
    c = np.bincount(map_hw.ravel(), minlength=n).astype(float)
    return 100 * c / c.sum()                       # area % per class, per month

# pct: (12, 7) stacked monthly percentages
# month-to-month change (pp):  np.diff(pct, axis=0)      -> (11, 7)
# total first->last change (pp): pct[-1] - pct[0]        -> (7,)
# transition matrix: count class(month1)->class(month12) per pixel, normalise to %

## Executed results — two contrasting locations

**Mannheim (urban, 49.49, 8.47) — Jan→Dec 2021.** A stable urban core: all classes
change < 1 pp and all transitions < 0.7% — the LSTM correctly reports a near-static
scene instead of inventing change. Dynamic World agreement **74.2%**.

**Flevoland (farmland, 52.52, 5.50) — Jan→Dec 2021.** Real seasonal dynamics:

| Class | Jan | Dec | Change |
|---|---|---|---|
| Trees | 53.2% | 56.4% | +3.18 pp |
| Built-up | 27.0% | 24.8% | −2.16 pp |
| Grassland | 9.8% | 8.4% | −1.38 pp |

Top transitions: Built-up→Trees 2.3%, Grassland→Trees 1.6%, Grassland→Cropland 0.6%
— largely **seasonal** (summer leaf-on raises radar volume scattering). Dynamic World
agreement **76.0%**.

**Advanced outputs (also produced per location):** change-detection map, confidence /
uncertainty map, transition-matrix heatmap. Figures saved per run under
`prototype_out_<tag>/`.

*Run it:* `python3 prototype.py --lat 49.4875 --lon 8.4660 --start-month 1 --year 2021 --tag mannheim`
*or launch the UI:* `python3 app.py`